In [0]:
from pyspark.sql.functions import *

In [0]:
batch2_file_path="/Volumes/grocery_delivery_project/source/initial_files/grocery_delivery_incremental.csv"

In [0]:
batch02_df=spark.read\
    .format("csv")\
        .option("header",True)\
            .option("inferSchema",True)\
                .csv(batch2_file_path)

In [0]:
display(batch02_df)

In [0]:
print("batch 2 total count:",batch02_df.count())

In [0]:
batch02_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in batch02_df.columns
]).show()

In [0]:
batch02_df.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
batch02_df.join(
    batch02_df.groupBy("order_id")
        .count()
        .filter(col("count") > 1)
        .select("order_id"),
    on="order_id",
    how="inner"
).display()

In [0]:
batch02_df.select([
    sum((trim(col(c)) == "").cast("int")).alias(c)
    for c in batch02_df.columns
]).show()

In [0]:
batch02_df.filter(col("unit_price") < 0).display()

In [0]:
batch02_df.filter(col("unit_price") < 0).count()

In [0]:
batch02_df.filter(
    try_to_date(col("order_date"), "M/d/yyyy").isNull() &
    col("order_date").isNotNull()
).display()

In [0]:
invalid_records = (
    (col("customer_name") == " ") |(col("customer_name").isNull())|
    (col("city").isNull()) |
    (col("unit_price") < 0) |
    (try_to_date(col("order_date"), "M/d/yyyy").isNull() &
     col("order_date").isNotNull()) |
    (col("order_id").isNull())|
    (col("delivery_status").isNull()) |
    (col("category").isNull())|
    (col("quantity") < 0)
)

In [0]:
bad_records=batch02_df.filter(
    invalid_records
)

In [0]:
bad_records.count()

In [0]:
bad_records.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema","true")\
    .saveAsTable("grocery_delivery_project.bronze.incremental_bad_records")

In [0]:
good_records=batch02_df.filter(
    ~invalid_records
)

In [0]:
print("total records:",batch02_df.count())
print("good records:",good_records.count())
print("bad records:",bad_records.count())

In [0]:
if(good_records.count()+bad_records.count()==batch02_df.count()):
    print("pass")
else:
    print("fail")

In [0]:
good_records.groupBy(
    "order_id").count().filter(col("count")>1).show()

In [0]:
cleaned_batch02=good_records.dropDuplicates(["order_id"])
display(cleaned_batch02)

In [0]:
cleaned_batch02.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("grocery_delivery_project.bronze.clean_batch02")